# Q&A RAG Pipeline

Build a RAG pipeline using the customer-support dataset to answer incoming customer questions. Vector DB: Chroma (local). Embeddings: sentence-transformers. LLM: Groq.

In [11]:
import sys
!{sys.executable} -m pip install chromadb sentence-transformers groq datasets pandas python-dotenv

## Setup Chroma DB and Embeddings

In [12]:
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import pandas as pd
import os

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.PersistentClient(path='./chroma_db')
collection_name = 'customer_support_kb'

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

collection = chroma_client.create_collection(name=collection_name)
print('ChromaDB initialized')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22410.82it/s]


ChromaDB initialized


## Populate Knowledge Base

In [13]:
dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
df = pd.DataFrame(dataset['train'])

# Use subset for quicker initialization in notebook (e.g., first 5000)
df_subset = df.head(5000)
texts = df_subset['response'].tolist()
ids = [str(i) for i in range(len(texts))]
metadata = [{'instruction': str(inst)} for inst in df_subset['instruction']]

print('Computing embeddings...')
embeddings = embedding_model.encode(texts, show_progress_bar=True)

print('Adding to vector database...')
collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadata,
    ids=ids
)
print('Knowledge base populated.')

Computing embeddings...


Batches: 100%|██████████| 157/157 [00:03<00:00, 47.96it/s]


Adding to vector database...
Knowledge base populated.


## Retrieval & Generation

In [14]:
def retrieve_context(query, top_k=3):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    return results['documents'][0]

from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

def generate_rag_response(user_message, detected_sentiment='neutral'):
    api_key = os.environ.get('GROQ_API_KEY')
    if not api_key:
        return "Error: GROQ_API_KEY environment variable not set."
    
    client = Groq(api_key=api_key)
    retrieved_chunks = retrieve_context(user_message)
    context_str = "\n".join(retrieved_chunks)

    system_prompt = f"""You are a helpful, professional customer support assistant
for an online retailer. Answer the customer's question using ONLY
the information in the retrieved support responses below. If the
customer sounds frustrated ({detected_sentiment}), acknowledge
that before answering. If the retrieved context does not cover
the question, say so honestly and offer to escalate to a human
agent rather than guessing."""

    user_prompt = f"Context:\n{context_str}\n\nCustomer question: \"{user_message}\""

    try:
        chat_completion = client.chat.completions.create(
            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
            model="mixtral-8x7b-32768",
            temperature=0.0
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Generation error: {str(e)}"

print('Test Response:')
print(generate_rag_response("Where is my package?", "neutral"))

Test Response:
Generation error: Error code: 400 - {'error': {'message': 'The model `mixtral-8x7b-32768` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
